# 실험 노트북
*1차 챌린지 · 전통적 신호 처리 기반 잡음 제거*

`denoise.py`의 현재 제출 코드를 Validation에서 평가하고, 잡음 종류·SNR별로 어디서 점수가 낮은지 찾습니다.
개선 실험은 이 노트북에서 하고, 좋아진 내용은 `denoise.py`에 반영한 뒤 `python evaluate.py`로 점수가 같은지 확인합니다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from pesq import pesq
from tqdm.auto import tqdm

# 저장소 루트(denoise.py가 있는 폴더)를 찾는다.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "denoise.py").is_file())
sys.path.insert(0, str(ROOT))

import denoise as submission   # 현재 제출 코드

pd.set_option("display.width", 160)
plt.rcParams["font.family"] = ["AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"]  # 한글 폰트 (맥/윈도우)
plt.rcParams["axes.unicode_minus"] = False
print("ROOT =", ROOT)

## 1. 평가 함수

`evaluate(fn, manifest)`는 `fn(noisy, sr)` 형태의 함수를 받아 파일별 SNR과 PESQ-WB를 계산합니다. Clean은 점수 계산에만 쓰고 `fn`에는 Noisy만 전달합니다.

In [ ]:
def snr_db(clean, est):
    error = np.sum((clean - est) ** 2)
    return float(np.inf if error == 0 else 10 * np.log10(np.sum(clean ** 2) / error))


def load_manifest(split="validation"):
    return pd.read_csv(ROOT / split / "samples.csv", encoding="utf-8-sig")


def evaluate(fn, manifest, desc="evaluate"):
    rows = []
    for r in tqdm(manifest.itertuples(), total=len(manifest), desc=desc):
        clean, sr = sf.read(ROOT / r.clean_path, dtype="float64")
        noisy, _ = sf.read(ROOT / r.noisy_path, dtype="float64")
        est = fn(noisy, sr)
        rows.append({
            "segment_id": r.segment_id,
            "noise_category": r.noise_category,
            "noise_difficulty": r.noise_difficulty,
            "target_snr_db": r.target_snr_db,
            "snr_noisy": snr_db(clean, noisy), "snr_denoised": snr_db(clean, est),
            "pesq_noisy": pesq(sr, clean, noisy, "wb"), "pesq_denoised": pesq(sr, clean, est, "wb"),
        })
    df = pd.DataFrame(rows)
    df["delta_pesq"] = df.pesq_denoised - df.pesq_noisy
    df["delta_snr"] = df.snr_denoised - df.snr_noisy
    return df


def summarize(df, by=None):
    cols = ["snr_noisy", "snr_denoised", "delta_snr", "pesq_noisy", "pesq_denoised", "delta_pesq"]
    if by is None:
        return df[cols].mean().round(4).to_frame("mean").T
    out = df.groupby(by)[cols].mean()
    out.insert(0, "n", df.groupby(by).size())
    return out.round(3)

## 2. 현재 제출 코드의 Validation 점수

`python evaluate.py`와 같은 값(PESQ 1.5984, SNR 13.1196)이 나와야 합니다.

In [ ]:
val = load_manifest("validation")
base = evaluate(submission.denoise, val, desc="validation")
summarize(base)

## 3. 어디서 약한가

`pesq_denoised`가 낮은 곳이 개선 후보이고, `delta_pesq`가 작거나 음수인 곳은 처리가 오히려 도움이 안 되는 곳입니다.

In [ ]:
print("잡음 종류별 (PESQ가 낮은 순)")
by_noise = summarize(base, "noise_category").sort_values("pesq_denoised")
display(by_noise)

In [ ]:
print("목표 SNR별")
display(summarize(base, "target_snr_db"))
print("난이도별")
display(summarize(base, "noise_difficulty"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = by_noise.index
axes[0].barh(order, by_noise.pesq_noisy, alpha=0.5, label="noisy")
axes[0].barh(order, by_noise.pesq_denoised, alpha=0.8, label="denoised")
axes[0].set_xlabel("PESQ-WB"); axes[0].set_title("잡음 종류별 PESQ"); axes[0].legend()

by_snr = summarize(base, "target_snr_db")
axes[1].plot(by_snr.index, by_snr.pesq_noisy, "o-", label="noisy")
axes[1].plot(by_snr.index, by_snr.pesq_denoised, "o-", label="denoised")
axes[1].set_xlabel("target SNR (dB)"); axes[1].set_ylabel("PESQ-WB")
axes[1].set_title("SNR별 PESQ"); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. 실험 기록

알고리즘이나 파라미터를 바꿀 때마다 아래 표에 점수를 남겨 두면 AI 활용 내역서의 "변경 전후 비교"에 그대로 쓸 수 있습니다.

| 실험 | 변경 내용 | PESQ-WB | SNR (dB) |
| :--- | :--- | ---: | ---: |
| baseline | Wiener Filter (제공 베이스라인과 동일) | 1.5984 | 13.1196 |
|  |  |  |  |

In [ ]:
# 실험 예시: 함수를 만들어 evaluate()에 넘기면 base와 같은 방식으로 비교할 수 있다.
# def my_denoise(noisy, sr):
#     ...
#     return enhanced
# exp = evaluate(my_denoise, val, desc="my_denoise")
# summarize(exp)

## 5. 직접 검증: 듣고, 스펙트로그램 보기

점수가 오른 것과 소리가 좋아진 것은 다를 수 있으므로, 파일을 골라 처리 전후를 직접 듣고 비교합니다.
점수가 많이 오른 파일과 오히려 나빠진 파일을 함께 확인하세요.

In [ ]:
# 가장 많이 좋아진 파일 5개 / 오히려 나빠진 파일 5개
cols = ["segment_id", "noise_category", "target_snr_db", "pesq_noisy", "pesq_denoised", "delta_pesq"]
print("가장 많이 좋아진 파일")
display(base.sort_values("delta_pesq", ascending=False)[cols].head(5))
print("오히려 나빠진 파일")
display(base.sort_values("delta_pesq")[cols].head(5))

In [ ]:
from IPython.display import Audio, display
from scipy import signal


def inspect_sample(segment_id, fn=None):
    """Clean / Noisy / Denoised 의 스펙트로그램과 재생 버튼을 보여준다. (세 그림의 색 기준 동일)"""
    fn = fn or submission.denoise
    r = val[val.segment_id == segment_id].iloc[0]
    clean, sr = sf.read(ROOT / r.clean_path)
    noisy, _ = sf.read(ROOT / r.noisy_path)
    est = fn(noisy, sr)
    items = [("Clean", clean), ("Noisy", noisy), ("Denoised", est)]

    specs = []
    for _, x in items:
        f, t, S = signal.spectrogram(x, sr, nperseg=512, noverlap=384)
        specs.append(10 * np.log10(S + 1e-12))
    vmax = max(s.max() for s in specs)
    vmin = vmax - 80

    print(f"{segment_id} | {r.noise_category} | target SNR {r.target_snr_db} dB")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, (name, _), s in zip(axes, items, specs):
        ax.pcolormesh(t, f, s, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(name)
        ax.set_xlabel("time (s)")
    axes[0].set_ylabel("Hz")
    plt.tight_layout()
    plt.show()
    for name, x in items:
        print(name)
        display(Audio(x, rate=sr))

In [ ]:
# 위 표에서 segment_id를 골라 실행하세요. (예: 가장 많이 좋아진 파일)
inspect_sample(base.sort_values("delta_pesq").iloc[-1].segment_id)

# 오히려 나빠진 파일
# inspect_sample(base.sort_values("delta_pesq").iloc[0].segment_id)

# 직접 고른 파일
# inspect_sample("speaker015_utt01_seg0001")

### 관찰 기록

들은 대로, 본 대로 적어 두세요. AI 활용 내역서의 "분석 및 고찰"에 쓰입니다.

| segment_id | 잡음 종류 / SNR | PESQ (전 → 후) | 청취한 느낌 | 스펙트로그램에서 본 것 |
| :--- | :--- | :--- | :--- | :--- |
|  |  |  |  |  |